In [ ]:
import pandas as pd
import json
import re

# --- 1. CONFIGURATION ---
maintenance_order_path = "data/mo_migrations.csv"
paths = {
    "RSD": "D:/Santai Coding/Pradigma - Digital Maintenance Portal/Data Migration/Validation Data/responses_rsd.csv",
    "PSD": "D:/Santai Coding/Pradigma - Digital Maintenance Portal/Data Migration/Validation Data/responses_psd.csv",
    "SNC": "D:/Santai Coding/Pradigma - Digital Maintenance Portal/Data Migration/Validation Data/responses_snc.csv",
    "TNM": "D:/Santai Coding/Pradigma - Digital Maintenance Portal/Data Migration/Validation Data/responses_tnm.csv"
}

# --- 2. THE TRANSFORMATION FUNCTION ---
def transform_and_log_offset(d):
    if not d or not isinstance(d, dict): return d
    
    car_bogie_to_global = {
        ('eca1', 1): 1, ('eca1', 2): 2,
        ('ica2', 1): 3, ('ica2', 2): 4,
        ('ica3', 1): 5, ('ica3', 2): 6,
        ('eca4', 1): 7, ('eca4', 2): 8
    }
    
    bogie_to_standard_car = {
        1: 'eca1', 2: 'eca1', 3: 'ica2', 4: 'ica2',
        5: 'ica3', 6: 'ica3', 7: 'eca4', 8: 'eca4'
    }
    
    new_dict = {}
    for k, v in d.items():
        match = re.match(r'(.*?)\.(.*?)\.bogie(\d+)\.(.*?)\.(.*?)\.(.*)', k)
        if match:
            main_prefix, old_car, local_b_str, component, pos, suffix = match.groups()
            local_bnum = int(local_b_str)
            car_key = old_car.lower()
            
            global_bnum = car_bogie_to_global.get((car_key, local_bnum), local_bnum)
            standard_car = bogie_to_standard_car.get(global_bnum, car_key)
            
            is_odd = global_bnum % 2 != 0
            
            # --- THE FIX: Only apply the gearbox/brake swap if the component is top/bottom ---
            comp_lower = component.lower()
            if "top" in comp_lower or "bottom" in comp_lower:
                is_top = "top" in comp_lower
                
                # Apply position mapping
                if is_top:
                    pos_map = {'a': 'bottom', 'b': 'right', 'c': 'left'}
                else:
                    pos_map = {'a': 'left', 'b': 'right', 'c': 'bottom'}
                new_pos = pos_map.get(pos, pos)

                # Apply component mapping
                if is_odd:
                    new_comp = "gearbox" if is_top else "brake"
                else:
                    new_comp = "brake" if is_top else "gearbox"
            else:
                # PRESERVE load_wheel and any other unmapped components
                new_comp = component 
                new_pos = pos
            # ---------------------------------------------------------------------------------

            new_key = f"{main_prefix}.{standard_car}.bogie{global_bnum}.{new_comp}.{new_pos}.{suffix}"
            new_dict[new_key] = v
        else:
            new_dict[k] = v
    return new_dict

# --- 3. SPECIAL PROCESSING FOR RSD ---
print("Processing RSD with Bogie Offsets...")
df_rsd = pd.read_csv(paths["RSD"])

for col in ['tyre_pressure', 'tyre_wear']:
    if col in df_rsd.columns:
        # Convert to Dict
        df_rsd[col] = df_rsd[col].apply(
            lambda x: json.loads(x) if pd.notnull(x) and str(x).strip() not in ["", "{}"] else None
        )
        # Apply Logic
        df_rsd[col] = df_rsd[col].apply(transform_and_log_offset)
        # Convert back to String for concatenation
        df_rsd[col] = df_rsd[col].apply(lambda x: json.dumps(x) if x is not None else "{}")

df_startend = pd.read_excel("D:/Santai Coding/Pradigma - Digital Maintenance Portal/Coding/python.notebook.extraction/technician_pm/output/rsd_startend.xlsx")
df_startend.rename(columns={'workorder': 'workorder_no'}, inplace=True)
cols_to_merge = ['workorder_no', 'start_date', 'end_date']
df_rsd = pd.merge(
    df_rsd, 
    df_startend[cols_to_merge], 
    on='workorder_no',
    how='left'
)

# --- 4. COMBINE ALL RESPONSE CSVS ---
df_list = [df_rsd] # Start with our cleaned RSD
for label, path in paths.items():
    if label == "RSD": continue # Skip raw RSD since we added the cleaned version
    try:
        temp_df = pd.read_csv(path)
        df_list.append(temp_df)
    except Exception as e:
        print(f"Error loading {label}: {e}")

df_responses_raw = pd.concat(df_list, ignore_index=True)

# --- 5. MERGE JSON COLUMNS TO TOP LEVEL ---
meta_cols = ['id', 'workorder_no', 'modified_at', 'filename', 'start_date', 'end_date']
json_cols = [col for col in df_responses_raw.columns if col not in meta_cols]

def merge_all_json_to_top_level(row):
    master_json = {}
    for col in json_cols:
        val = row[col]
        if isinstance(val, str) and val.strip() not in ["{}", ""]:
            try:
                data = json.loads(val)
                if isinstance(data, dict):
                    master_json.update(data)
            except (json.JSONDecodeError, TypeError):
                continue
    return json.dumps(master_json)

df_responses_raw['responses'] = df_responses_raw.apply(merge_all_json_to_top_level, axis=1)

responses_cleaned = df_responses_raw.sort_values('modified_at').drop_duplicates(subset=['workorder_no'], keep='last')
responses_cleaned = responses_cleaned[['workorder_no', 'responses', 'filename', 'start_date', 'end_date']]

# --- 6. FINAL MASTER MERGE ---
df_mo = pd.read_csv(maintenance_order_path)
final_migration_df = pd.merge(df_mo, responses_cleaned, on='workorder_no', how='left')
final_migration_df['responses'] = final_migration_df['responses'].fillna("{}")

rename_dict = {
    "eca1.bogie1": "eca1.bogie1",
    "eca1.bogie2": "eca1.bogie2",
    "ica2.bogie1": "ica2.bogie3",
    "ica2.bogie2": "ica2.bogie4",
    "ica3.bogie1": "ica3.bogie5",
    "ica3.bogie2": "ica3.bogie6",
    "eca4.bogie1": "eca4.bogie7",
    "eca4.bogie2": "eca4.bogie8",
}

def rename_bogie_keys(responses_str):
    if not responses_str or responses_str.strip() in ["{}", ""]:
        return responses_str
    
    try:
        data = json.loads(responses_str)
    except (json.JSONDecodeError, TypeError):
        return responses_str
    
    new_data = {}
    for key, value in data.items():
        new_key = key
        for old_segment, new_segment in rename_dict.items():
            if old_segment in key:
                new_key = key.replace(old_segment, new_segment)
                break  # Only one segment will match per key
        new_data[new_key] = value
    
    return json.dumps(new_data)

final_migration_df['responses'] = final_migration_df['responses'].apply(rename_bogie_keys)

# --- 7. FORM TEMPLATE MAPPING ---
print("Processing Form Templates...")

def assign_form_template(filename):
    filename_str = str(filename) if pd.notnull(filename) else ""
    dept = filename_str.split('_')[0]
    
    if dept == "RS":
        interval = filename_str.split('_')[2]
        if interval == "WEK":
            return "RSD_PM_Weekly"
        elif interval == "MTH":
            return "RSD_PM_Monthly"
        elif interval == "QTR":
            return "RSD_PM_Quarterly"
        elif interval == "HYL":
            return "RSD_PM_HalfYearly"
        elif interval == "YRL":
            return "RSD_PM_Yearly"
    elif dept == "PS" or dept == "SC" or dept == "TN": return filename_str.split('_')[3]
        
final_migration_df['form_template'] = final_migration_df['filename'].apply(assign_form_template)

# --- 8. OPERATION ACTIVITY ---
def format_activity(val):
    if pd.isnull(val):
        return None
    return str(int(val)).zfill(4)

final_migration_df['oper_activity'] = final_migration_df['oper_activity'].apply(format_activity)

cols_final = [col for col in final_migration_df.columns if col != 'responses'] + ['responses']
final_migration_df = final_migration_df[cols_final]

# --- 9. EXPORT ---
dummy_df = final_migration_df[final_migration_df['work_centre'] == 'RSDM'].head(20)
dummy_df.sort_values('workorder_no').to_csv("final_migration_data_dummy.csv", index=False)
print(f"Dummy Migration file generated: {len(dummy_df)} records.")

final_migration_df.sort_values('workorder_no').to_csv("final_migration_data.csv", index=False)
print(f"Migration file generated: {len(final_migration_df)} records.")

Processing RSD with Bogie Offsets...
Processing Form Templates...
Dummy Migration file generated: 20 records.
Migration file generated: 21200 records.


In [2]:
import numpy as np

# Ensure sorting column is correct
sort_col = 'workorder_no'

# Filter datasets
rsd_df = final_migration_df[final_migration_df['work_centre'] == 'RSDM'].sort_values(by=sort_col)
ws10_df = final_migration_df[final_migration_df['work_centre'] == 'WS10'].sort_values(by=sort_col)
ws20_df = final_migration_df[final_migration_df['work_centre'] == 'WS20'].sort_values(by=sort_col)
tnm_df = final_migration_df[final_migration_df['work_centre'] == 'TNM1'].sort_values(by=sort_col)

for df in [rsd_df, ws10_df, ws20_df, tnm_df]:
    string_cols = df.select_dtypes(include=['object', 'string']).columns
    for col in string_cols:
        if df[col].str.contains('\n', na=False).any():
            print(f"Column '{col}' in {df['work_centre'].iloc[0]} contains newline characters.")
        df[col] = df[col].str.replace('\n', ' ', regex=False)

# --- Export WS10, WS20, TNM ---
ws10_df.to_csv('migration_psd.csv', index=False)
ws20_df.to_csv('migration_snc.csv', index=False)
tnm_df.to_csv('migration_tnm.csv', index=False)

print("✅ WS10, WS20, TNM exported successfully")

# --- Split RSD into 4 parts ---
rsd_splits = np.array_split(rsd_df, 4)

for i, split_df in enumerate(rsd_splits, start=1):
    filename = f'migration_rsd_batch{i}.csv'
    split_df.to_csv(filename, index=False)
    print(f"✅ Exported {filename} with {len(split_df)} rows")

print("✅ All RSD files exported successfully")

Column 'work_request' in RSDM contains newline characters.
Column 'work_request' in WS20 contains newline characters.
✅ WS10, WS20, TNM exported successfully
✅ Exported migration_rsd_batch1.csv with 3264 rows


d:\Santai Coding\Pradigma - Digital Maintenance Portal\Coding\python.notebook.extraction\venv\Lib\site-packages\numpy\_core\fromnumeric.py:57: FutureWarning: 'DataFrame.swapaxes' is deprecated and will be removed in a future version. Please use 'DataFrame.transpose' instead.
  return bound(*args, **kwds)


✅ Exported migration_rsd_batch2.csv with 3263 rows
✅ Exported migration_rsd_batch3.csv with 3263 rows
✅ Exported migration_rsd_batch4.csv with 3263 rows
✅ All RSD files exported successfully


In [3]:
# 1. Select only columns that contain strings or objects
# string_cols = ws20_df.select_dtypes(include=['object', 'string']).columns

# for col in string_cols:
#     # 2. Add na=False to handle missing values safely
#     if ws20_df[col].str.contains('\n', na=False).any():
#         print(f"Column '{col}' contains newline characters.")
    
#     # ws20_df[col] = ws20_df[col].str.replace('\n', ' ', regex=False)

In [4]:
# df_ups = ws10_df[ws10_df['form_template'] == 'StationInspection']
# df_ups.to_csv('PSD_StationInspection.csv', index=False)
# df_ups.shape